# SignalScout: API walkthrough

This notebook drives the SignalScout REST API end to end, on the data from a real pipeline run (4 news sources → 55 scored leads).

**To reproduce:** `docker compose up -d db`, then in `backend/` run `uv run signalscout run` and `uv run uvicorn signalscout.api.main:app --port 8000`. Interactive OpenAPI docs are at http://localhost:8000/docs.

Sections:
1. Funnel stats: how much noise was filtered
2. "Who should I call today?": filtered, ranked leads
3. Why a lead scored what it did
4. Contact data with validation
5. AI first-touch email
6. Changing the Ideal Customer Profile re-ranks everything instantly
7. CRM-ready CSV export
8. Scraper health

In [1]:
import io, json
import httpx
import pandas as pd

API = "http://localhost:8000/api"
api = httpx.Client(base_url=API, timeout=120)
pd.set_option("display.max_colwidth", 80, "display.width", 200)

def money(v):
    return "—" if not v else f"${v/1e9:.1f}B" if v >= 1e9 else f"${v/1e6:.1f}M"

## 1. Funnel stats

In [2]:
stats = api.get("/stats").json()
pd.DataFrame([
    ("Articles scraped", stats["articles"]),
    ("Rejected as not a company raise", stats["filtered_out"]),
    ("Funding articles", stats["funding_articles"]),
    ("Duplicate rounds merged across outlets", stats["merged_duplicates"]),
    ("Active leads", stats["leads"]),
    ("Hot leads (score 80+)", stats["hot_leads"]),
    ("Leads with a verified email", stats["with_verified_email"]),
    ("Capital raised in the last 7 days", money(stats["raised_this_week_usd"])),
], columns=["metric", "value"])

,metric,value
0,Articles scraped,50
1,Rejected as not a company raise,22
2,Funding articles,25
3,Duplicate rounds merged across outlets,3
4,Active leads,55
5,Hot leads (score 80+),22
6,Leads with a verified email,15
7,Capital raised in the last 7 days,$4.2B


## 2. "Who should I call today?"

Series A or Seed rounds from the last 14 days, scoring 70+, with an email address, ranked by score.

In [3]:
leads = api.get("/leads", params={"stage": ["Seed", "Series A"], "days": 14, "min_score": 70,
                                  "has_email": True, "sort": "score"}).json()
print(f"{leads['total']} leads match")
pd.DataFrame([{
    "score": l["score"], "company": l["name"], "sector": l["sector"], "city": l["hq_city"],
    "round": l["latest_round"]["round_type"], "amount": money(l["latest_round"]["amount_usd"]),
    "announced": l["latest_round"]["announced_on"], "best email": l["best_email"]["email"],
    "confidence": l["best_email"]["confidence"],
} for l in leads["items"]])

3 leads match


,score,company,sector,city,round,amount,announced,best email,confidence
0,81,Andera,SaaS / Software,NaN,Series A,$37.0M,2026-10-05,team@andera.ai,high
1,77,Kapta Space,Aerospace / Defense,Seattle,Series A,$24.0M,2026-10-09,info@kaptaspace.com,high
2,73,Ghost,Hardware / Robotics,San Francisco,Seed,$11.0M,2026-10-05,support@ghost.ai,high


## 3. Why did the top lead score what it did?

In [4]:
top = leads["items"][0]
lead = api.get(f"/leads/{top['id']}").json()
print(f"{lead['name']}: {lead['score']}/100   ({lead['description']})\n")
print("Breakdown:", lead["score_breakdown"])
print("\nReasons:")
for r in lead["score_reasons"]:
    print("  •", r)
ev = lead["funding_events"][0]
print(f"\nRound: {ev['round_type']} {money(ev['amount_usd'])} on {ev['announced_on']}, led by {', '.join(ev['lead_investors']) or 'n/a'}")
print("Use of funds:", ev["use_of_funds"])
print("Sources:", *[f"{a['source']}: {a['title']}" for a in lead["articles"]], sep="\n  ")

Andera: 81/100   (Andera automates corporate audit and compliance tasks.)

Breakdown: {'fit': 30, 'size': 15, 'timing': 24, 'reachability': 12}

Reasons:
  • Raised $37M Series A 7 days ago
  • Target sector: SaaS / Software
  • Target stage: Series A
  • Actively hiring - budget is being deployed
  • Verified email on company domain

Round: Series A $37.0M on 2026-10-05, led by Lightspeed
Use of funds: None
Sources:
  techcrunch: 5 startups that caught VCs’ attention at the latest PearX demo day


## 4. Contact data with validation

Emails are MX-checked and labelled `high` (company domain and accepts mail), `medium`, or `low`. Phones are normalised to E.164. People come from the press coverage.

In [5]:
print("Website:", lead["website"], f"(source: {lead['website_source']})", "| data quality:", lead["data_quality"], "/ 100")
for label, rows in [("Emails", lead["emails"]), ("Phones", lead["phones"]), ("People", lead["people"])]:
    print(f"{label}:")
    display(pd.DataFrame(rows)) if rows else print("  none published")

Website: https://www.andera.ai/ (source: article) | data quality: 60 / 100
Emails:


,type,email,mx_valid,confidence,same_domain
0,role,team@andera.ai,True,high,True


Phones:
  none published
People:
  none published


## 5. AI first-touch email

Generated from the funding facts plus the user's own offer (the `pitch` in the ICP), then cached on the lead.

In [6]:
draft = api.post(f"/leads/{top['id']}/email-draft").json()
print("Subject:", draft["subject"], "\n")
print(draft["body"], "\n")
print("Send to:", draft["contact_hint"])

Subject: Scaling Andera outbound after Series A 

Andera recently raised $37,000,000 in a Series A round led by Lightspeed. Companies at this stage typically need to accelerate pipeline generation as they expand their sales team. We help newly funded startups scale their go-to-market with outbound lead generation. Are you currently building out your outbound infrastructure? 

Send to: Head of Sales or VP of Growth, as they own outbound GTM execution.


## 6. Changing the ICP re-ranks every lead instantly

Example: a sales team that only sells to healthcare companies. Scoring is pure computation over stored data, so there's no re-scraping and no LLM calls.

In [7]:
original = api.get("/icp").json()["icp"]
before = {l["name"]: l["score"] for l in api.get("/leads", params={"limit": 500}).json()["items"]}

healthcare = {**original, "sectors": ["Healthcare / Biotech"], "stages": ["Seed", "Series A", "Series B", "Undisclosed"]}
print("Re-scored:", api.put("/icp", json=healthcare).json())
after = api.get("/leads", params={"limit": 8}).json()["items"]
display(pd.DataFrame([{"company": l["name"], "sector": l["sector"], "score now": l["score"],
                       "score before": before[l["name"]]} for l in after]))

api.put("/icp", json=original)  # restore the default profile
print("ICP restored")

Re-scored: {'rescored': 55}


,company,sector,score now,score before
0,Biotia,Healthcare / Biotech,99,89
1,Parakeet Health,Healthcare / Biotech,91,81
2,PreciseDx,Healthcare / Biotech,90,80
3,Polyphron,Healthcare / Biotech,88,88
4,Wally,Healthcare / Biotech,80,80
5,True,Fintech,80,100
6,Kapta Space,Aerospace / Defense,77,77
7,Presto,SaaS / Software,76,86


ICP restored


## 7. CRM-ready CSV export (HubSpot/Salesforce-style headers)

In [8]:
csv_text = api.get("/export.csv", params={"min_score": 80}).text
export = pd.read_csv(io.StringIO(csv_text))
print(f"{len(export)} hot leads exported, columns:\n{list(export.columns)}")
export.head(5)[["Company Name", "Website", "Lead Score", "Last Round", "Amount (USD)", "Primary Email", "Why Now"]]

22 hot leads exported, columns:
['Company Name', 'Website', 'Industry', 'City', 'Country', 'Lead Score', 'Data Quality', 'Last Round', 'Amount (USD)', 'Announced', 'Lead Investors', 'Use of Funds', 'Primary Email', 'Email Confidence', 'Phone', 'LinkedIn', 'Key People', 'Why Now', 'Status']


,Company Name,Website,Lead Score,Last Round,Amount (USD),Primary Email,Why Now
0,True,https://true.ai,100,Series B,2500000,support@true.ai,Raised $2M Series B 5 days ago | Target sector: Fintech | In target geograph...
1,GUDEA,https://www.gudea.ai/,92,Seed,7000000,NaN,Raised $7M Seed 3 days ago | Target sector: AI / ML | In target geography (M...
2,Melius,https://www.melius.com/,92,Series A,20000000,NaN,Raised $20M Series A 4 days ago | Target sector: AI / ML | In target geograp...
3,Meticulous,https://meticulous.ai,92,Series A,15000000,NaN,Raised $15M Series A 4 days ago | Target sector: SaaS / Software | In target...
4,Stuut,https://www.stuut.ai/,92,Series B,52500000,NaN,Raised $52M Series B 5 days ago | Target sector: Fintech | In target geograp...


## 8. Scraper health (Spidermon-style)

In [9]:
pd.DataFrame(api.get("/health/sources").json())[["source", "status", "feed_entries", "items_new", "items_skipped_seen", "items_dropped", "message"]]

,source,status,feed_entries,items_new,items_skipped_seen,items_dropped,message
0,techcrunch,ok,40,3,17,0,None
1,alleywatch,ok,10,0,10,0,None
2,prnewswire,ok,20,0,11,0,None
3,geekwire,ok,35,0,6,0,None
